# Regression Analysis - Credit Card Data

**Goal:** predict `Amount` (a number) from the features `V1` to `V28`.

Techniques: Simple Linear Regression, Multiple Linear Regression, Polynomial Regression.

We use `creditcard_clean.csv` from preprocessing. `Amount` is scaled there, so predictions are in scaled units. Low R2 is expected because the V features were not made to predict Amount.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
data = pd.read_csv('creditcard_clean.csv')
data.head()

## 1. Correlation with Amount

Find which features are most related to `Amount`. The strongest one is used for simple linear regression.

In [ ]:
v_cols = ['V' + str(i) for i in range(1, 29)]   # V1 to V28

corr = data[v_cols + ['Amount']].corr()['Amount'].drop('Amount')
corr_sorted = corr.abs().sort_values(ascending=False)   # strength, ignoring + or -
print(corr_sorted.head(5))

best_feat = corr_sorted.index[0]
top5 = corr_sorted.index[:5].tolist()
print('Best feature:', best_feat)
print('Top 5 features:', top5)

In [ ]:
sns.heatmap(data[top5 + ['Amount']].corr(), annot=True)
plt.show()

## 2. Train-test split

Model learns from `x_train, y_train` (80%). We check it on `x_test, y_test` (20%), which it has never seen.

In [ ]:
x = data[v_cols]
y = data['Amount']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, x_test.shape)

## 3. Simple Linear Regression (one feature)

`Amount = intercept + slope * feature`

In [ ]:
model = LinearRegression()
model.fit(x_train[[best_feat]], y_train)

y_pred = model.predict(x_test[[best_feat]])

print('Intercept:', model.intercept_)
print('Slope:', model.coef_[0])
print('R2 score:', r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(x_test[best_feat][:2000], y_test[:2000], color='blue', label='Actual')
plt.plot(x_test[best_feat], y_pred, color='red', label='Predicted Line')

plt.title('Simple Linear Regression')
plt.xlabel(best_feat)
plt.ylabel('Amount')
plt.legend()
plt.show()

In [ ]:
# predict for a new value
new_value = np.array([[1.5]])
print(model.predict(new_value))

**R2 score** tells how much of the variation in Amount the model explains. 1 is perfect, 0 means the model is no better than guessing the average.

## 4. Multiple Linear Regression (all V features)

`Amount = intercept + b1*V1 + b2*V2 + ... + b28*V28`

In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print('Intercept:', model.intercept_)
print('R2 score:', r2_score(y_test, y_pred))

# coefficient of each feature
pd.Series(model.coef_, index=v_cols).round(3)

In [ ]:
# predict for one row of test data and compare with the real value
print('Predicted:', model.predict(x_test.iloc[[0]]))
print('Actual   :', y_test.iloc[0])

## 5. Polynomial Regression

Adds squared, cubed... terms so the model can fit a curve. We use the best single feature so the curve can be plotted.

- `fit_transform` on train data = learn and apply
- `transform` on test data = only apply what was learned

In [ ]:
x1_train = x_train[[best_feat]]
x1_test = x_test[[best_feat]]

# try different degrees
for d in [1, 2, 3, 4]:
    poly = PolynomialFeatures(degree=d)
    x_train_trans = poly.fit_transform(x1_train)
    x_test_trans = poly.transform(x1_test)

    model = LinearRegression()
    model.fit(x_train_trans, y_train)

    r2_train = r2_score(y_train, model.predict(x_train_trans))
    r2_test = r2_score(y_test, model.predict(x_test_trans))
    print('Degree', d, '| R2 train:', round(r2_train, 4), '| R2 test:', round(r2_test, 4))

**Overfitting:** if R2 train keeps going up but R2 test does not, the model is memorising the training data, not learning a real pattern.

In [ ]:
# plot the degree 2 curve
poly = PolynomialFeatures(degree=2)
x_train_trans = poly.fit_transform(x1_train)
x_test_trans = poly.transform(x1_test)

model = LinearRegression()
model.fit(x_train_trans, y_train)

x_new = np.linspace(x1_train[best_feat].min(), x1_train[best_feat].max(), 200).reshape(200, 1)
xnew_poly = poly.transform(x_new)
y_new = model.predict(xnew_poly)

plt.figure(figsize=(10, 5))
plt.plot(x1_train[best_feat][:2000], y_train[:2000], 'b.')
plt.plot(x1_test[best_feat][:2000], y_test[:2000], 'g.')
plt.plot(x_new, y_new, 'r-')
plt.xlabel(best_feat)
plt.ylabel('Amount')
plt.title('Polynomial Regression (degree 2)')
plt.show()

In [ ]:
# polynomial with top 5 features (degree 2) vs plain linear on the same 5
x5_train = x_train[top5]
x5_test = x_test[top5]

model = LinearRegression()
model.fit(x5_train, y_train)
print('Linear top 5     R2:', r2_score(y_test, model.predict(x5_test)))

poly = PolynomialFeatures(degree=2)
x5_train_trans = poly.fit_transform(x5_train)
x5_test_trans = poly.transform(x5_test)

model = LinearRegression()
model.fit(x5_train_trans, y_train)
print('Polynomial top 5 R2:', r2_score(y_test, model.predict(x5_test_trans)))

## 6. Errors (residuals)

**Residual** = actual - predicted. For a good model most residuals are small and centred around 0.

In [ ]:
# using multiple linear regression
model = LinearRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

residuals = y_test - y_pred

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(y_test[:2000], y_pred[:2000], s=6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=80)
plt.title('Residuals')

plt.tight_layout()
plt.show()

## Conclusion
- R2 is low for all models, so the V features explain only a small part of Amount.
- Polynomial terms improve the fit very little over linear regression, so the extra complexity is not worth it.
- Regression is practice here. The main task for this dataset is classification (fraud or not).